# 04 — FRQI, NEQR and hybrid quantum-classical experiments

This notebook works from a labelled MSD patient slice. FRQI and NEQR state representations are tested at 16×16 and 32×32. Qiskit Aer ideal/noisy simulations are separate from the exact NumPy statevector round-trip. State preparation (`initialize`) is explicitly counted as a costly data-loading step; it is not free QRAM.

**Practical guidance:** start with 16×16. NEQR 32×32 requires 18 qubits; its dense statevector has 262,144 complex amplitudes. The 18-qubit Aer simulation/transpilation may be expensive on a CPU-only backend.

In [ ]:
from pathlib import Path
import sys, json, numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = ROOT / "data/raw/Task01_BrainTumour"

In [ ]:
from qmi.data.msd_dataset import read_msd_metadata, discover_msd_cases, load_msd_case, case_to_slices
from qmi.data.preprocessing import resize_slice
from qmi.quantum.representations import (encode_frqi_state, encode_neqr_state, decode_frqi_state, decode_neqr_state)
from qmi.metrics import reconstruction_metrics
meta = read_msd_metadata(DATA_ROOT)
cases = discover_msd_cases(DATA_ROOT)
split_path = ROOT / "outputs/reports/msd_patient_split.json"
if not split_path.exists(): raise FileNotFoundError("Run Notebook 02 first to create the patient-level split.")
split = json.loads(split_path.read_text(encoding="utf-8"))
case_by_id = {item.case_id: item for item in cases}
if not split.get("test"): raise ValueError("Saved split has no held-out test patients.")
case = case_by_id[split["test"][0]]  # keep the hybrid demonstration on a held-out patient
volumes, labels = load_msd_case(case, meta)
slices = case_to_slices(volumes, labels, max_slices=8, tumor_first=True, return_indices=True)
positive = [item for item in slices if np.any(item[2] > 0)]
if not positive: raise ValueError("Selected held-out patient has no tumour-positive slice.")
z, _, _ = max(positive, key=lambda item: int(np.count_nonzero(item[2])))
print("Case:", case.case_id, "slice:", z)

In [ ]:
# Exact representation encoding/decoding is measured independently of Qiskit Aer.
rows = []
for resolution in (16, 32):
    modalities, truth = resize_slice(volumes[:, z], labels[z], resolution)
    image = np.rint(modalities[0] * 255).astype(np.uint8)
    for name, encoder, decoder in [("FRQI", encode_frqi_state, decode_frqi_state), ("NEQR", encode_neqr_state, decode_neqr_state)]:
        state = encoder(image)
        reconstructed = decoder(state, resolution)
        scores = reconstruction_metrics(image, reconstructed, data_range=255)
        rows.append({"resolution": resolution, "representation": name, "qubits": int(np.log2(len(state))), **scores})
pd.DataFrame(rows)

In [ ]:
from qmi.quantum.simulation import profile_circuit
image16 = np.rint(resize_slice(volumes[:, z], labels[z], 16)[0][0] * 255).astype(np.uint8)
for rep in ("FRQI", "NEQR"):
    print(rep, profile_circuit(image16, rep, max_qubits=16))

In [ ]:
# Real hybrid path requires the corresponding MSD-trained RF model.
from qmi.models import load_model
from qmi.hybrid import run_hybrid_path
RESOLUTION = 16
MODEL_PATH = ROOT / "artifacts" / f"msd_rf_{RESOLUTION}.joblib"
if not MODEL_PATH.exists():
    raise FileNotFoundError("Run Notebook 02 first to train the MSD model.")
modalities, truth = resize_slice(volumes[:, z], labels[z], RESOLUTION)
model, model_metadata = load_model(MODEL_PATH)
result = run_hybrid_path(modalities, model, "FRQI", true_labels=truth, simulation_mode="ideal", device="CPU")
print("Reconstruction:", result.reconstruction)
print("Active-region macro Dice:", result.segmentation["macro_dice_active"])
print("All-region macro Dice (empty/empty = 1):", result.segmentation["macro_dice"])
print("Simulator metadata:", result.simulator_metadata)

In [ ]:
# Optional follow-up: noisy shot-based simulation (16x16 recommended as the first noise experiment).
# result_noisy = run_hybrid_path(modalities, model, "FRQI", true_labels=truth,
#     simulation_mode="noisy", shots=50_000, device="CPU")
# print(result_noisy.reconstruction, result_noisy.segmentation)

## Interpretation and reporting

- FRQI intensity is encoded by a rotation angle; decoding must invert the sine-squared measurement probability.
- NEQR stores 8-bit grayscale values in computational-basis qubits.
- Reconstruction results must include MSE, PSNR and SSIM when SSIM is defined.
- Segmentation results must include WT/TC/ET metrics on the same held-out labelled slice.
- Report qubits, logical circuit depth, transpiled depth/gate counts when transpilation completes, shot count, noise parameters, device, and wall time.
- Simulator runs are classical simulations of quantum circuits. They do not establish an advantage over classical algorithms.